Additional Features:
 - Excess style momentum
 - Excess style volatility (against benchmark)
 - Excess style kurtosis (against benchmark)
 - Excess style skewness (against benchmark)
 - Relative momentum (factors relative to each other, trend scan - categorical features)
 - Short-technical indicators (remove)
 - Medium technical indicators (remove)
 - Omega ration (remove)


In [1]:
# run script from the data_input file
import os
os.chdir('C:/Users/p528552/OneDrive - South African Reserve Bank/Documents/1. MEng - Data Science/1. Project_2025/Data/factor_timing/data_input')

In [2]:
# Import libraries:
import numpy as np
import pandas as pd
from scipy.stats import kurtosis, skew


In [3]:
# Import data:
df_factors = pd.read_csv("factor_returns.csv", parse_dates=['Date'])
df_factors = df_factors.set_index('Date')

df_vol = pd.read_csv("local_global_vol.csv", parse_dates=['Date'])
df_vol = df_vol.set_index('Date')

df = pd.concat([df_factors,df_vol], axis=1, join='inner')
df = df.dropna()

print(df_vol)
#print(df.dtypes)


                SPX     JALSH    VIX    MOVE  SAVIT40  VIX3M  VIX6M  VIX1Y
Date                                                                      
1998-10-01  1017.01   4695.37  40.95  124.22      NaN    NaN    NaN    NaN
1998-11-01  1098.67   5357.66  28.05  121.06      NaN    NaN    NaN    NaN
1998-12-01  1163.63   5202.07  26.01   87.22      NaN    NaN    NaN    NaN
1999-01-01  1229.23   5025.85  24.42  121.13      NaN    NaN    NaN    NaN
1999-02-01  1279.64   5427.55  26.25   87.16      NaN    NaN    NaN    NaN
...             ...       ...    ...     ...      ...    ...    ...    ...
2024-09-01  5648.40  83749.86  15.00  107.77    16.07  17.43  18.88  20.29
2024-10-01  5762.48  86548.42  16.73   94.61    15.25  19.22  20.62  21.85
2024-11-01  5705.45  85384.82  23.16  135.18    17.32  22.64  22.61  22.73
2024-12-01  6032.38  84510.44  13.51   95.22    16.78  16.26  18.15  19.74
2024-12-03  6047.15  85731.51  13.34   97.81    16.17  16.34  18.05  19.81

[316 rows x 8 columns]


In [7]:
def generate_factor_features(df, macro_df=None, window_short=4, window_med=12, window_long=26):
    """
    Generate time-series, technical, and macro-interaction features from factor returns.

    """

    feat = pd.DataFrame(index=df.index)
    
    for col in df.columns:
        series = df[col]
        name = col.lower()
        
        # ========== 1️⃣ Momentum / Value / Size Features ==========
        feat[f"{name}_ret_lag1"] = series.shift(1)
        feat[f"{name}_ret_lag4"] = series.shift(4)
        feat[f"{name}_ret_lag12"] = series.shift(12)

        feat[f"{name}_roll_mean_4w"] = series.rolling(window_short).mean()
        feat[f"{name}_roll_mean_12w"] = series.rolling(window_med).mean()
        feat[f"{name}_roll_std_12w"] = series.rolling(window_med).std()
        feat[f"{name}_roll_sharpe_12w"] = (
            feat[f"{name}_roll_mean_12w"] / feat[f"{name}_roll_std_12w"]
        )

        # Consecutive positive/negative streak
        streak = series.gt(0).astype(int)
        feat[f"{name}_streak"] = streak.groupby((streak != streak.shift()).cumsum()).cumsum()

        # Relative performance (pairwise spreads)
        for other in df.columns:
            if other != col:
                feat[f"{name}_vs_{other.lower()}"] = series - df[other]

        # ========== 2️⃣ Sentiment & Technical Indicators ==========
        # Rolling z-score
        feat[f"{name}_zscore_12w"] = (series - series.rolling(window_med).mean()) / series.rolling(window_med).std()

        # RSI
        delta = series.diff()
        up = np.where(delta > 0, delta, 0)
        down = np.where(delta < 0, -delta, 0)
        roll_up = pd.Series(up, index=series.index).rolling(window_med).mean()
        roll_down = pd.Series(down, index=series.index).rolling(window_med).mean()
        rs = roll_up / roll_down
        feat[f"{name}_rsi_12w"] = 100 - (100 / (1 + rs))

        # MACD and Signal
        ema_short = series.ewm(span=12, adjust=False).mean()
        ema_long = series.ewm(span=26, adjust=False).mean()
        macd = ema_short - ema_long
        signal = macd.ewm(span=9, adjust=False).mean()
        feat[f"{name}_macd"] = macd
        feat[f"{name}_macd_signal"] = signal
        feat[f"{name}_macd_hist"] = macd - signal

        # Bollinger deviation
        roll_mean = series.rolling(window_med).mean()
        roll_std = series.rolling(window_med).std()
        feat[f"{name}_bollinger_z"] = (series - roll_mean) / roll_std

        # Skewness and Kurtosis (higher-order moments)
        feat[f"{name}_skew_26w"] = series.rolling(window_long).skew()
        feat[f"{name}_kurt_26w"] = series.rolling(window_long).kurt()

        # Rolling drawdown
        cum = (1 + series).cumprod()
        running_max = cum.cummax()
        feat[f"{name}_drawdown"] = (cum / running_max) - 1

    # ========== 3️⃣ Macro-Economic Interaction Features ==========
    if macro_df is not None:
        aligned = df.join(macro_df, how="inner")
        for f in df.columns:
            for m in macro_df.columns:
                feat[f"{f.lower()}_corr_{m.lower()}"] = (
                    aligned[f].rolling(window_med).corr(aligned[m])
                )
                feat[f"{f.lower()}_beta_{m.lower()}"] = (
                    aligned[f].rolling(window_med).cov(aligned[m]) /
                    aligned[m].rolling(window_med).var()
                )

    # ========== 4️⃣ Cross-Factor Interaction ==========
    for i, f1 in enumerate(df.columns):
        for f2 in df.columns[i+1:]:
            f1_low, f2_low = f1.lower(), f2.lower()
            feat[f"{f1_low}_{f2_low}_corr"] = df[f1].rolling(window_med).corr(df[f2])
            feat[f"{f1_low}_{f2_low}_vol_ratio"] = (
                df[f1].rolling(window_med).std() / df[f2].rolling(window_med).std()
            )

    return feat

# ===========================
# Example usage
# ===========================
df = pd.read_csv("factor_returns.csv", index_col='Date', parse_dates=True)
# macro_df = pd.read_csv("macro_features.csv", index_col='Date', parse_dates=True)
factor_relative_features = generate_factor_features(df)
factor_relative_features.dropna().head()


,momentum_ret_lag1,momentum_ret_lag4,momentum_ret_lag12,momentum_roll_mean_4w,momentum_roll_mean_12w,momentum_roll_std_12w,momentum_roll_sharpe_12w,momentum_streak,momentum_vs_value,momentum_vs_quality,...,quality_bollinger_z,quality_skew_26w,quality_kurt_26w,quality_drawdown,momentum_value_corr,momentum_value_vol_ratio,momentum_quality_corr,momentum_quality_vol_ratio,value_quality_corr,value_quality_vol_ratio
Date,,,,,,,,,,,,,,,,,,,,,
2007-07-16,-0.000025,-0.002226,-0.010031,-0.001750,0.001392,0.018199,0.076487,0,-0.005453,0.047420,...,-2.106179,-0.599294,0.714538,-0.051885,0.862396,1.076012,-0.294510,0.783635,-0.294750,0.728277
2007-07-23,-0.003532,-0.024960,0.013544,-0.007241,-0.003647,0.022412,-0.162722,0,0.000765,-0.039958,...,-0.211463,-0.490521,0.619534,-0.058489,0.910480,1.060714,-0.187642,0.964886,-0.181697,0.909658
2007-07-30,-0.046923,0.021515,-0.001166,-0.013750,-0.003926,0.022399,-0.175290,0,0.005634,0.006938,...,-0.402728,-0.462680,0.528204,-0.069276,0.909532,1.057988,-0.182342,0.963581,-0.172537,0.910768
2007-08-06,-0.004520,-0.000025,-0.018871,-0.015120,-0.002813,0.021916,-0.128344,0,-0.000086,0.032766,...,-1.282604,-0.418345,0.251542,-0.104898,0.924813,1.035399,-0.123326,0.875319,-0.158484,0.845393
2007-08-13,-0.005508,-0.003532,0.014535,-0.026926,-0.008254,0.025092,-0.328939,0,-0.009931,-0.104218,...,1.831384,-0.157755,0.108705,-0.057043,0.935314,1.087145,-0.421550,0.826777,-0.383194,0.760503


In [8]:
# Save output into csv file:
factor_relative_features.to_csv('factor_relative_features.csv', index=True)

In [ ]:
print(factor_relative_features)

            Momentum     Value   Quality
Date                                    
2007-01-01       NaN       NaN  0.000000
2007-01-08       NaN  0.013676  0.000000
2007-01-15       NaN  0.016786  0.000000
2007-01-22  0.000357  0.010732  0.014931
2007-01-29  0.021934  0.011995 -0.005972
...              ...       ...       ...
2025-09-08       NaN  0.000000  0.000000
2025-09-15       NaN  0.000000  0.000000
2025-09-22       NaN  0.000000  0.000000
2025-09-29       NaN  0.000000  0.000000
2025-10-06       NaN  0.000000  0.000000

[980 rows x 3 columns]


In [ ]:
# Geometric and arithmetc means:

def generate_momentum_features(df, cols, windows=[1, 3, 6, 12]):

    df = df.copy()
    
    for col in cols:
        for w in windows:
            # Rolling arithmetic mean
            #df[f"{col}_rollmean_{w}m"] = df[col].rolling(window=w).mean()

            # Rolling geometric mean
            df[f"{col}_geommean_{w}m"] = (
                np.exp(np.log(df[col] + 1).rolling(window=w).mean()) - 1
            )

    return df


cols = ['Momentum','Value','Quality','SPX','JALSH']
df_mom = generate_momentum_features(df, cols)


df_mom.columns

Index(['Benchmark', 'Momentum', 'Value', 'Quality', 'SPX', 'JALSH', 'VIX',
       'MOVE', 'SAVIT40', 'VIX3M', 'VIX6M', 'VIX1Y', 'Momentum_geommean_1m',
       'Momentum_geommean_3m', 'Momentum_geommean_6m', 'Momentum_geommean_12m',
       'Momentum_geommean_24m', 'Value_geommean_1m', 'Value_geommean_3m',
       'Value_geommean_6m', 'Value_geommean_12m', 'Value_geommean_24m',
       'Quality_geommean_1m', 'Quality_geommean_3m', 'Quality_geommean_6m',
       'Quality_geommean_12m', 'Quality_geommean_24m', 'SPX_geommean_1m',
       'SPX_geommean_3m', 'SPX_geommean_6m', 'SPX_geommean_12m',
       'SPX_geommean_24m', 'JALSH_geommean_1m', 'JALSH_geommean_3m',
       'JALSH_geommean_6m', 'JALSH_geommean_12m', 'JALSH_geommean_24m'],
      dtype='object')

In [6]:
df_mom_features = df_mom[['Momentum_geommean_1m',
       'Momentum_geommean_3m', 'Momentum_geommean_6m', 'Momentum_geommean_12m',
       'Momentum_geommean_24m', 'Value_geommean_1m', 'Value_geommean_3m',
       'Value_geommean_6m', 'Value_geommean_12m', 'Value_geommean_24m',
       'Quality_geommean_1m', 'Quality_geommean_3m', 'Quality_geommean_6m',
       'Quality_geommean_12m', 'Quality_geommean_24m', 'SPX_geommean_1m',
       'SPX_geommean_3m', 'SPX_geommean_6m', 'SPX_geommean_12m',
       'SPX_geommean_24m', 'JALSH_geommean_1m', 'JALSH_geommean_3m',
       'JALSH_geommean_6m', 'JALSH_geommean_12m', 'JALSH_geommean_24m']]

df_mom_features.to_csv("momentum_features.csv", index=True)

In [7]:
# Excess momentum, volatility, kurtosis and skewness:

window = 6

for style in styles:
    # Excess Style Momentum (Cumulative excess return)
    df[f'{style}_excess_momentum'] = df[f'{style}_excess'].rolling(window).sum()
    

    # Excess Style Volatility
    df[f'{style}_excess_volatility'] = df[f'{style}_excess'].rolling(window).std()
    

    # Excess Style Kurtosis
    df[f'{style}_excess_kurtosis'] = df[f'{style}_excess'].rolling(window).apply(kurtosis, raw=True)

    # Excess Style Skewness
    df[f'{style}_excess_skewness'] = df[f'{style}_excess'].rolling(window).apply(skew, raw=True)
    
    


NameError: name 'styles' is not defined

In [ ]:
# Compare factors risk to benchmark risk:

# Rolling risk stats for benchmark
df['benchmark_vol'] = df['Benchmark'].rolling(window).std()
df['benchmark_kurtosis'] = df['Benchmark'].rolling(window).apply(kurtosis, raw=True)
df['benchmark_skewness'] = df['Benchmark'].rolling(window).apply(skew, raw=True)

for style in styles:
    df[f'{style}_relative_volatility'] = df[f'{style}'].rolling(window).std() / df['benchmark_vol']
    df[f'{style}_relative_kurtosis'] = df[f'{style}'].rolling(window).apply(kurtosis, raw=True) / df['benchmark_kurtosis']
    df[f'{style}_relative_skewness'] = df[f'{style}'].rolling(window).apply(skew, raw=True) / df['benchmark_skewness']


In [ ]:
# Technical indicators:
for style in styles:
    df[f'{style}_momentum_1M'] = df[style].shift(1)
    df[f'{style}_momentum_3M'] = df[style].rolling(3).sum()
    df[f'{style}_momentum_6M'] = df[style].rolling(6).sum()
    df[f'{style}_momentum_12M'] = df[style].rolling(12).sum()
    df[f'{style}_momentum_24M'] = df[style].rolling(24).sum()


In [ ]:
# Relative factor performance:

df = df.drop(['Benchmark','Momentum','Value','Quality','Momentum_excess','Value_excess','Quality_excess',
              'benchmark_vol','benchmark_kurtosis','benchmark_skewness'],axis=1)
print(df.columns)


Index(['Momentum_excess_momentum', 'Momentum_excess_volatility',
       'Momentum_excess_kurtosis', 'Momentum_excess_skewness',
       'Value_excess_momentum', 'Value_excess_volatility',
       'Value_excess_kurtosis', 'Value_excess_skewness',
       'Quality_excess_momentum', 'Quality_excess_volatility',
       'Quality_excess_kurtosis', 'Quality_excess_skewness',
       'Momentum_relative_volatility', 'Momentum_relative_kurtosis',
       'Momentum_relative_skewness', 'Value_relative_volatility',
       'Value_relative_kurtosis', 'Value_relative_skewness',
       'Quality_relative_volatility', 'Quality_relative_kurtosis',
       'Quality_relative_skewness', 'Momentum_momentum_1M',
       'Momentum_momentum_3M', 'Momentum_momentum_6M', 'Momentum_momentum_12M',
       'Momentum_momentum_24M', 'Value_momentum_1M', 'Value_momentum_3M',
       'Value_momentum_6M', 'Value_momentum_12M', 'Value_momentum_24M',
       'Quality_momentum_1M', 'Quality_momentum_3M', 'Quality_momentum_6M',
       